[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/54_activation_checkpointing_solution.ipynb)

# 🟡 Solution: Implement Activation Checkpointing

Reference solution for activation checkpointing — recompute activations during backward to save memory.

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch
import torch.utils.checkpoint as cp

In [ ]:
# ✅ SOLUTION

def checkpoint_sequential(fns, x):
    for fn in fns:
        x = cp.checkpoint(fn, x, use_reentrant=False)
    return x

In [ ]:
torch.manual_seed(0)
layers = [torch.nn.Linear(16, 16) for _ in range(4)]
x = torch.randn(4, 16, requires_grad=True)

# Checkpointed forward
out_cp = checkpoint_sequential(layers, x)

# Naive sequential forward
x2 = x.detach().requires_grad_(True)
out_naive = x2
for layer in layers:
    out_naive = layer(out_naive)

print("Outputs match:", torch.allclose(out_cp, out_naive, atol=1e-5))

# Confirm gradients flow through checkpointed path
out_cp.sum().backward()
print("Gradient flows (x.grad is not None):", x.grad is not None)

In [ ]:
from torch_judge import check
check("activation_checkpointing")